In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir('/content')

!rm -rf /content/graduation-p
!git clone https://github.com/berk0vic/graduation-p.git /content/graduation-p

os.chdir('/content/graduation-p/notebooks')
sys.path.insert(0, '/content/graduation-p')

!rm -rf /content/graduation-p/data
!ln -s /content/drive/MyDrive/graduation-p/data /content/graduation-p/data

import torch
TORCH = torch.__version__.split('+')[0]
CUDA = torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
!pip install -q pyg-lib torch-scatter torch-sparse torch-cluster torch-geometric -f https://data.pyg.org/whl/torch-{TORCH}+cu{CUDA}.html
!pip install -q xgboost shap

# 03 — Model Training (IEEE-CIS)
**Dual-Branch architecture:** HeteroGAT (graph embeddings) + VAE (anomaly detection) → fraud classifier

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import time
from torch_geometric.loader import NeighborLoader

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Device: {device}')
print(f'CWD: {os.getcwd()}')

cfg_path = '../configs/default.yaml' if os.path.exists('../configs/default.yaml') else 'configs/default.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)
print('Model:', {k: v for k, v in cfg['model'].items()})
print('Training:', {k: v for k, v in cfg['training'].items()})

## 1. Heterogeneous Graph Construction

In [ ]:
from src.graph.builder import build_hetero_graph

train_txn = pd.read_csv('../data/raw/ieee_cis/train_transaction.csv')
train_id = pd.read_csv('../data/raw/ieee_cis/train_identity.csv')
df = train_txn.merge(train_id, on='TransactionID', how='left')
print(f'Rows: {len(df):,}, Columns: {len(df.columns)}')

data = build_hetero_graph(df, dataset='ieee_cis')

y = data['transaction'].y
n_fraud = y.sum().item()
n_total = len(y)
print(f'Fraud: {n_fraud:,} / {n_total:,} ({100*n_fraud/n_total:.2f}%)')

os.makedirs('../data/processed/ieee_cis', exist_ok=True)
torch.save(data, '../data/processed/ieee_cis/hetero_graph_v3.pt')

raw_txn_features = data['transaction'].x.clone()
print(f'Raw txn features: {raw_txn_features.shape}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Left: Graph structure summary ---
edge_counts = {f"{s}→{t}": data[s, r, t].edge_index.shape[1]
               for s, r, t in data.edge_types}
bars = axes[0].barh(list(edge_counts.keys()), list(edge_counts.values()), color='#9b59b6')
axes[0].set_xlabel('Edge Count')
axes[0].set_title('Graph Structure (GCN / GAT input)')
for bar in bars:
    axes[0].text(bar.get_width() + max(edge_counts.values())*0.01,
                 bar.get_y() + bar.get_height()/2,
                 f'{int(bar.get_width()):,}', va='center', fontsize=9)

# --- Center: Node counts ---
node_info = {ntype: data[ntype].num_nodes for ntype in data.node_types}
colors_n = ['#e74c3c', '#2ecc71', '#3498db']
bars2 = axes[1].bar(list(node_info.keys()), list(node_info.values()), color=colors_n[:len(node_info)])
axes[1].set_ylabel('Count')
axes[1].set_title('Node Types in Heterogeneous Graph')
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(node_info.values())*0.01,
                 f'{int(bar.get_height()):,}', ha='center', fontsize=10)

# --- Right: Feature dimensions per node type ---
feat_dims = {ntype: data[ntype].x.shape[1] for ntype in data.node_types}
bars3 = axes[2].bar(list(feat_dims.keys()), list(feat_dims.values()), color=colors_n[:len(feat_dims)])
axes[2].set_ylabel('Feature Dimension')
axes[2].set_title('Feature Dimensions per Node Type')
for bar in bars3:
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{int(bar.get_height())}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 2. Train / Val / Test Split (Stratified)

In [ ]:
from sklearn.model_selection import train_test_split

y_np = y.numpy()
indices = np.arange(n_total)

train_idx, temp_idx = train_test_split(indices, test_size=0.3, stratify=y_np, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=y_np[temp_idx], random_state=42)

train_mask = torch.zeros(n_total, dtype=torch.bool)
val_mask = torch.zeros(n_total, dtype=torch.bool)
test_mask = torch.zeros(n_total, dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

data['transaction'].train_mask = train_mask
data['transaction'].val_mask = val_mask
data['transaction'].test_mask = test_mask

print(f'Train: {train_mask.sum().item():,} (fraud: {y[train_mask].sum().item():,})')
print(f'Val:   {val_mask.sum().item():,} (fraud: {y[val_mask].sum().item():,})')
print(f'Test:  {test_mask.sum().item():,} (fraud: {y[test_mask].sum().item():,})')

## 3. XGBoost Baseline (Tabular Only)

In [ ]:
from xgboost import XGBClassifier
from src.evaluation.metrics import compute_metrics, print_report

X_train = raw_txn_features[train_mask].numpy()
X_val = raw_txn_features[val_mask].numpy()
X_test = raw_txn_features[test_mask].numpy()
y_train = y[train_mask].numpy()
y_val = y[val_mask].numpy()
y_test = y[test_mask].numpy()

fraud_ratio = y_train.sum() / (len(y_train) - y_train.sum())

t0 = time.time()
xgb = XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    scale_pos_weight=1/fraud_ratio,
    eval_metric='aucpr', early_stopping_rounds=20,
    tree_method='hist', n_jobs=-1, random_state=42, verbosity=0,
)
xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_time = time.time() - t0
print(f'XGBoost training: {xgb_time:.1f}s')

xgb_scores = xgb.predict_proba(X_test)[:, 1]
xgb_metrics = compute_metrics(y_test, xgb_scores)
print_report(y_test, xgb_scores)

### What XGBoost Sees vs What Graph Models See

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

importances = xgb.feature_importances_
top_k = 20
top_idx = np.argsort(importances)[-top_k:]
ax.barh(range(top_k), importances[top_idx], color='#2ECC71')
ax.set_yticks(range(top_k))
ax.set_yticklabels([f'feature_{i}' for i in top_idx], fontsize=9)
ax.set_xlabel('Importance')
ax.set_title(f'XGBoost — Top {top_k} Feature Importances (Tabular Only, No Graph)')
plt.tight_layout()
plt.show()

## 4. GCN Baseline (Homogeneous Graph)

In [ ]:
from src.models.baselines import GCNBaseline
from src.training.losses import focal_loss
from collections import defaultdict

acct_txn_edges = data['account', 'initiates', 'transaction'].edge_index
txn_x = data['transaction'].x

acct_to_txns = defaultdict(list)
for a, t in zip(acct_txn_edges[0].tolist(), acct_txn_edges[1].tolist()):
    acct_to_txns[a].append(t)

src_list, dst_list = [], []
for txns in acct_to_txns.values():
    if len(txns) > 1:
        for i in range(len(txns)):
            for j in range(i+1, min(i+5, len(txns))):
                src_list.extend([txns[i], txns[j]])
                dst_list.extend([txns[j], txns[i]])

homo_edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
print(f'Homogeneous graph: {txn_x.shape[0]:,} nodes, {homo_edge_index.shape[1]:,} edges')

gcn = GCNBaseline(in_channels=txn_x.shape[1], hidden=64, out=32).to(device)
opt_gcn = torch.optim.Adam(gcn.parameters(), lr=0.001, weight_decay=1e-4)

txn_x_dev = txn_x.to(device)
homo_edges_dev = homo_edge_index.to(device)
y_dev = y.float().to(device)
train_mask_dev = train_mask.to(device)
val_mask_dev = val_mask.to(device)

best_val = float('inf')
patience_cnt = 0

t0 = time.time()
for epoch in range(1, 101):
    gcn.train()
    opt_gcn.zero_grad()
    logits = gcn(txn_x_dev, homo_edges_dev)
    loss = focal_loss(logits[train_mask_dev], y_dev[train_mask_dev])
    loss.backward()
    opt_gcn.step()

    gcn.eval()
    with torch.no_grad():
        vl = focal_loss(gcn(txn_x_dev, homo_edges_dev)[val_mask_dev], y_dev[val_mask_dev]).item()
    if vl < best_val:
        best_val = vl; patience_cnt = 0
        best_gcn = {k: v.cpu().clone() for k, v in gcn.state_dict().items()}
    else:
        patience_cnt += 1
        if patience_cnt >= 15:
            print(f'Early stopping epoch {epoch}')
            break
    if epoch % 25 == 0:
        print(f'  Epoch {epoch:3d} | train: {loss.item():.4f} | val: {vl:.4f}')

gcn.load_state_dict(best_gcn)
gcn_time = time.time() - t0
print(f'GCN training: {gcn_time:.1f}s')

gcn.eval()
with torch.no_grad():
    gcn_scores = torch.sigmoid(gcn(txn_x_dev, homo_edges_dev)).cpu().numpy()
gcn_test_scores = gcn_scores[test_mask.numpy()]

gcn_metrics = compute_metrics(y_test, gcn_test_scores)
print_report(y_test, gcn_test_scores)

## 5. Hybrid GAT+VAE (Dual-Branch)

| Branch | Input | Output |
|--------|-------|--------|
| **HeteroGAT** | Graph structure + node features | `h_t` (graph embeddings) |
| **VAE** | Raw transaction features | Reconstruction error (anomaly score) |
| **Classifier** | `[h_t ∥ recon_err]` | Fraud probability |

In [ ]:
from src.models.hybrid_model import HybridGATVAE
from src.training.losses import focal_loss, reconstruction_loss, kl_divergence

in_channels = {ntype: data[ntype].x.shape[1] for ntype in data.node_types}
raw_txn_dim = data['transaction'].x.shape[1]

model = HybridGATVAE(
    metadata=data.metadata(),
    in_channels=in_channels,
    raw_txn_dim=raw_txn_dim,
    gat_hidden=cfg['model']['gat_hidden'],
    gat_out=cfg['model']['gat_out'],
    gat_heads=cfg['model']['gat_heads'],
    gat_layers=cfg['model']['gat_layers'],
    vae_latent=cfg['model']['vae_latent'],
    vae_hidden=cfg['model']['vae_hidden'],
    dropout=cfg['model']['dropout'],
).to(device)

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Feature dims: {in_channels}')
print(f'Raw txn dim: {raw_txn_dim}')

### Phase 1: VAE Pre-training (Normal Transactions Only)

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

normal_train_mask = train_mask & (y == 0)
normal_features = raw_txn_features[normal_train_mask]
print(f'Normal train samples: {len(normal_features):,}')

for p in model.gat.parameters(): p.requires_grad = False
for p in model.classifier.parameters(): p.requires_grad = False
for p in model.recon_bn.parameters(): p.requires_grad = False
for p in model.vae.parameters(): p.requires_grad = True

vae_optimizer = torch.optim.Adam(model.vae.parameters(), lr=0.001, weight_decay=1e-4)
vae_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(vae_optimizer, patience=5, factor=0.5)

vae_dataset = TensorDataset(normal_features)
vae_loader = DataLoader(vae_dataset, batch_size=4096, shuffle=True)

phase1_history = []
t0 = time.time()

for epoch in range(1, 31):
    model.train()
    epoch_recon, epoch_kl, n_batches = 0, 0, 0

    for (batch_x,) in vae_loader:
        batch_x = batch_x.to(device)
        vae_optimizer.zero_grad()

        x_recon, mu, logvar = model.vae(batch_x)
        l_recon = reconstruction_loss(batch_x, x_recon)
        l_kl = kl_divergence(mu, logvar)
        loss = l_recon + 0.01 * l_kl

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.vae.parameters(), max_norm=1.0)
        vae_optimizer.step()

        epoch_recon += l_recon.item()
        epoch_kl += l_kl.item()
        n_batches += 1

    avg_recon = epoch_recon / n_batches
    avg_kl = epoch_kl / n_batches
    vae_scheduler.step(avg_recon)
    phase1_history.append({'epoch': epoch, 'recon': avg_recon, 'kl': avg_kl})

    if epoch % 10 == 0:
        print(f'  Epoch {epoch:3d} | recon: {avg_recon:.4f} | kl: {avg_kl:.4f}')

p1_time = time.time() - t0
print(f'Phase 1 done ({p1_time:.1f}s). Final recon: {phase1_history[-1]["recon"]:.4f}')

for p in model.parameters(): p.requires_grad = True

### Phase 2: End-to-End Training (NeighborLoader + Differential LR)

In [ ]:
batch_size = cfg['training']['batch_size']
num_neighbors = cfg['training']['num_neighbors']

train_loader = NeighborLoader(
    data,
    num_neighbors=[5, 3],
    batch_size=4096,
    input_nodes=('transaction', train_mask),
    shuffle=True,
)

val_loader = NeighborLoader(
    data,
    num_neighbors=num_neighbors,
    batch_size=batch_size * 2,
    input_nodes=('transaction', val_mask),
    shuffle=False,
)

print(f'Train loader: {len(train_loader)} batches')
print(f'Val loader: {len(val_loader)} batches')

sample_batch = next(iter(train_loader))
print(f'\nSample batch:')
print(f'  Transaction nodes: {sample_batch["transaction"].num_nodes}')
print(f'  Seed nodes: {sample_batch["transaction"].batch_size}')
print(f'  Account nodes: {sample_batch["account"].num_nodes}')
print(f'  Merchant nodes: {sample_batch["merchant"].num_nodes}')
for et in sample_batch.edge_types:
    print(f'  Edge {et}: {sample_batch[et].edge_index.shape[1]}')

In [ ]:
lr = cfg['training']['lr']
optimizer = torch.optim.Adam([
    {'params': model.gat.parameters(), 'lr': lr},
    {'params': model.vae.parameters(), 'lr': lr * 0.1},
    {'params': model.recon_bn.parameters(), 'lr': lr},
    {'params': model.classifier.parameters(), 'lr': lr * 2},
], weight_decay=cfg['training']['weight_decay'])

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

lambda1 = cfg['training']['lambda1']
lambda2 = cfg['training']['lambda2']
focal_alpha = cfg['training']['focal_alpha']
focal_gamma = cfg['training']['focal_gamma']
max_epochs = cfg['training']['epochs']
patience = cfg['training']['patience']

print(f'LR: GAT={lr}, VAE={lr*0.1}, Classifier={lr*2}')
print(f'Lambda1(recon)={lambda1}, Lambda2(kl)={lambda2}')
print(f'Focal: alpha={focal_alpha}, gamma={focal_gamma}')

phase2_history = []
best_val_loss = float('inf')
patience_counter = 0
best_state = None
t0 = time.time()

for epoch in range(1, max_epochs + 1):
    model.train()
    train_loss_sum, train_cls_sum, train_recon_sum = 0, 0, 0
    n_train_batches = 0

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        bs = batch['transaction'].batch_size
        raw_txn = batch['transaction'].x
        outputs = model(batch.x_dict, batch.edge_index_dict, raw_txn_features=raw_txn)

        logit = outputs['logit'][:bs]
        raw = outputs['raw_txn'][:bs]
        x_recon = outputs['x_recon'][:bs]
        mu = outputs['mu'][:bs]
        logvar = outputs['logvar'][:bs]
        targets = batch['transaction'].y[:bs].float()

        l_cls = focal_loss(logit, targets, gamma=focal_gamma, alpha=focal_alpha)
        l_recon = reconstruction_loss(raw, x_recon)
        l_kl = kl_divergence(mu, logvar)
        loss = l_cls + lambda1 * l_recon + lambda2 * l_kl

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        train_cls_sum += l_cls.item()
        train_recon_sum += l_recon.item()
        n_train_batches += 1

    avg_train = train_loss_sum / n_train_batches
    avg_cls = train_cls_sum / n_train_batches
    avg_recon = train_recon_sum / n_train_batches

    model.eval()
    val_loss_sum, n_val_batches = 0, 0

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            bs = batch['transaction'].batch_size
            raw_txn = batch['transaction'].x
            outputs = model(batch.x_dict, batch.edge_index_dict, raw_txn_features=raw_txn)

            logit = outputs['logit'][:bs]
            raw = outputs['raw_txn'][:bs]
            x_recon = outputs['x_recon'][:bs]
            mu = outputs['mu'][:bs]
            logvar = outputs['logvar'][:bs]
            targets = batch['transaction'].y[:bs].float()

            l_cls = focal_loss(logit, targets, gamma=focal_gamma, alpha=focal_alpha)
            l_recon = reconstruction_loss(raw, x_recon)
            l_kl = kl_divergence(mu, logvar)
            val_loss = l_cls + lambda1 * l_recon + lambda2 * l_kl

            val_loss_sum += val_loss.item()
            n_val_batches += 1

    avg_val = val_loss_sum / n_val_batches
    scheduler.step(avg_val)

    phase2_history.append({
        'epoch': epoch, 'train_loss': avg_train, 'val_loss': avg_val,
        'cls': avg_cls, 'recon': avg_recon,
    })

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    if epoch % 1 == 0:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'  Epoch {epoch:3d} | cls: {avg_cls:.4f} | recon: {avg_recon:.4f} | val: {avg_val:.4f} | lr: {lr_now:.6f}')

    if patience_counter >= patience:
        print(f'\nEarly stopping at epoch {epoch}')
        break

if best_state:
    model.load_state_dict(best_state)
    model.to(device)

p2_time = time.time() - t0
print(f'\nPhase 2 done ({p2_time:.1f}s). Best val loss: {best_val_loss:.4f}')

## 6. Test Results

In [ ]:
model.eval()

test_loader = NeighborLoader(
    data,
    num_neighbors=[-1, -1],
    batch_size=4096,
    input_nodes=('transaction', test_mask),
    shuffle=False,
)

all_scores = []
all_targets = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        bs = batch['transaction'].batch_size
        raw_txn = batch['transaction'].x
        outputs = model(batch.x_dict, batch.edge_index_dict, raw_txn_features=raw_txn)
        all_scores.append(outputs['fraud_score'][:bs].cpu())
        all_targets.append(batch['transaction'].y[:bs].cpu())

hybrid_test_scores = torch.cat(all_scores).numpy()
hybrid_test_y = torch.cat(all_targets).numpy()

hybrid_metrics = compute_metrics(hybrid_test_y, hybrid_test_scores)
print_report(hybrid_test_y, hybrid_test_scores)

fraud_mask_test = hybrid_test_y == 1
print(f'\nScore stats — fraud mean: {hybrid_test_scores[fraud_mask_test].mean():.4f}, normal mean: {hybrid_test_scores[~fraud_mask_test].mean():.4f}')

## 7. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['XGBoost', 'GCN Baseline', 'Hybrid GAT+VAE'],
    'F1 (Fraud)': [xgb_metrics['f1_fraud'], gcn_metrics['f1_fraud'], hybrid_metrics['f1_fraud']],
    'Recall': [xgb_metrics['recall_fraud'], gcn_metrics['recall_fraud'], hybrid_metrics['recall_fraud']],
    'Precision': [xgb_metrics['precision_fraud'], gcn_metrics['precision_fraud'], hybrid_metrics['precision_fraud']],
    'AUC-ROC': [xgb_metrics['roc_auc'], gcn_metrics['roc_auc'], hybrid_metrics['roc_auc']],
    'Avg Precision': [xgb_metrics['avg_precision'], gcn_metrics['avg_precision'], hybrid_metrics['avg_precision']],
})
print(results.to_string(index=False))

os.makedirs('../results/tables', exist_ok=True)
results.to_csv('../results/tables/ieee_cis_results_v2.csv', index=False)

In [ ]:
metrics_to_plot = ['F1 (Fraud)', 'Recall', 'Precision', 'AUC-ROC']
x = np.arange(len(metrics_to_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2ECC71', '#5DADE2', '#E74C3C']
for i, (_, row) in enumerate(results.iterrows()):
    vals = [row[m] for m in metrics_to_plot]
    bars = ax.bar(x + i*width - width, vals, width, label=row['Model'], color=colors[i])
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_ylabel('Score')
ax.set_title('Model Comparison — IEEE-CIS Dataset')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/model_comparison_v2.png', dpi=150)
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot([h['recon'] for h in phase1_history], label='Reconstruction', color='#E74C3C')
ax1.plot([h['kl'] for h in phase1_history], label='KL Divergence', color='#3498DB')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Phase 1: VAE Pre-training')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot([h['train_loss'] for h in phase2_history], label='Train Loss', color='#E74C3C')
ax2.plot([h['val_loss'] for h in phase2_history], label='Val Loss', color='#3498DB')
ax2.plot([h['recon'] for h in phase2_history], label='Recon Loss', color='#27AE60', linestyle='--')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Phase 2: End-to-End Training')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/training_loss_v2.png', dpi=150)
plt.show()

## 8. Score Distribution — What Each Model Captures

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Score distributions ---
bins = np.linspace(0, 1, 60)
for scores, y_true, name, color in [
    (xgb_scores, y_test, 'XGBoost', '#2ECC71'),
    (gcn_test_scores, y_test, 'GCN', '#5DADE2'),
    (hybrid_test_scores, hybrid_test_y, 'Hybrid GAT+VAE', '#E74C3C'),
]:
    axes[0].hist(scores[y_true == 0], bins=bins, alpha=0.35, color=color, label=f'{name} Normal')
    axes[0].hist(scores[y_true == 1], bins=bins, alpha=0.7, color=color, histtype='step', linewidth=2, label=f'{name} Fraud')
axes[0].set_xlabel('Fraud Score')
axes[0].set_ylabel('Count (log)')
axes[0].set_yscale('log')
axes[0].set_title('Score Distribution — Fraud vs Normal')
axes[0].legend(fontsize=7, ncol=2)

# --- ROC curves ---
for scores, y_true, name, color in [
    (xgb_scores, y_test, 'XGBoost', '#2ECC71'),
    (gcn_test_scores, y_test, 'GCN', '#5DADE2'),
    (hybrid_test_scores, hybrid_test_y, 'Hybrid GAT+VAE', '#E74C3C'),
]:
    fpr, tpr, _ = roc_curve(y_true, scores)
    axes[1].plot(fpr, tpr, color=color, linewidth=2, label=name)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# --- Precision-Recall curves ---
for scores, y_true, name, color in [
    (xgb_scores, y_test, 'XGBoost', '#2ECC71'),
    (gcn_test_scores, y_test, 'GCN', '#5DADE2'),
    (hybrid_test_scores, hybrid_test_y, 'Hybrid GAT+VAE', '#E74C3C'),
]:
    prec, rec, _ = precision_recall_curve(y_true, scores)
    axes[2].plot(rec, prec, color=color, linewidth=2, label=name)
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curves')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/score_distribution_comparison.png', dpi=150)
plt.show()

## 10. Complementarity Analysis — What Does the Graph Catch That XGBoost Misses?

XGBoost only sees tabular features (transaction amount, card info, etc.).  
Our Hybrid GAT+VAE also sees **graph structure** — who transacts with whom, shared merchants, identity links.

The key question: **Are there fraud cases that XGBoost misses but GNN catches?**

In [ ]:
# Get optimal thresholds for each model
xgb_thresh = xgb_metrics['threshold']
hybrid_thresh = hybrid_metrics['threshold']

# Binary predictions
xgb_pred = (xgb_scores >= xgb_thresh).astype(int)
hybrid_pred = (hybrid_test_scores >= hybrid_thresh).astype(int)

# Find the 4 groups
both_catch    = (xgb_pred == 1) & (hybrid_pred == 1) & (y_test == 1)  # both correct
only_xgb      = (xgb_pred == 1) & (hybrid_pred == 0) & (y_test == 1)  # only XGBoost catches
only_hybrid   = (xgb_pred == 0) & (hybrid_pred == 1) & (y_test == 1)  # only Hybrid catches
both_miss     = (xgb_pred == 0) & (hybrid_pred == 0) & (y_test == 1)  # both miss

total_fraud = (y_test == 1).sum()

print(f"Total fraud in test set: {total_fraud}")
print(f"  Both catch:        {both_catch.sum():5d}  ({100*both_catch.sum()/total_fraud:.1f}%)")
print(f"  Only XGBoost:      {only_xgb.sum():5d}  ({100*only_xgb.sum()/total_fraud:.1f}%)")
print(f"  Only Hybrid GNN:   {only_hybrid.sum():5d}  ({100*only_hybrid.sum()/total_fraud:.1f}%)")
print(f"  Both miss:         {both_miss.sum():5d}  ({100*both_miss.sum()/total_fraud:.1f}%)")

In [ ]:
# Venn-style bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Stacked bar showing who catches what ---
categories = ['Both Catch', 'Only XGBoost', 'Only Hybrid\nGAT+VAE', 'Both Miss']
counts = [both_catch.sum(), only_xgb.sum(), only_hybrid.sum(), both_miss.sum()]
colors = ['#2ECC71', '#5DADE2', '#E74C3C', '#95A5A6']

bars = axes[0].bar(categories, counts, color=colors, edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{count}\n({100*count/total_fraud:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[0].set_ylabel('Number of Fraud Cases')
axes[0].set_title('Fraud Detection Coverage — XGBoost vs Hybrid GAT+VAE')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# --- Right: Pie chart ---
axes[1].pie(counts, labels=categories, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 10})
axes[1].set_title('Share of Fraud Cases by Detection')

plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/complementarity_analysis.png', dpi=150)
plt.show()

print(f"\n>> Hybrid GAT+VAE catches {only_hybrid.sum()} fraud cases that XGBoost completely misses!")
print(f">> These are likely fraud rings or identity-linked patterns invisible to tabular models.")

### Why does the Hybrid model catch extra fraud?

Let's look at the **graph neighborhood** of fraud cases that only the Hybrid model catches.  
If these transactions are connected to other fraud nodes via shared accounts/merchants, that confirms the graph structure adds value.

In [ ]:
# Get test set transaction indices (global indices in the graph)
test_indices = torch.where(test_mask)[0].numpy()

# Map: which test-set positions are "only hybrid catches"
only_hybrid_positions = np.where(only_hybrid)[0]
only_hybrid_global = test_indices[only_hybrid_positions]

# Also get "both catch" for comparison
both_catch_positions = np.where(both_catch)[0]
both_catch_global = test_indices[both_catch_positions]

# Count fraud neighbors for each group
# Use account→transaction edges to find co-account transactions
acct_edges = data['account', 'initiates', 'transaction'].edge_index
fraud_set = set(torch.where(y == 1)[0].tolist())

def count_fraud_neighbors(txn_indices):
    """For each transaction, count how many fraud transactions share the same account."""
    # Build txn → account mapping
    txn_to_accts = {}
    for acct, txn in zip(acct_edges[0].tolist(), acct_edges[1].tolist()):
        if txn not in txn_to_accts:
            txn_to_accts[txn] = []
        txn_to_accts[txn].append(acct)
    
    # Build account → txns mapping
    acct_to_txns = {}
    for acct, txn in zip(acct_edges[0].tolist(), acct_edges[1].tolist()):
        if acct not in acct_to_txns:
            acct_to_txns[acct] = []
        acct_to_txns[acct].append(txn)
    
    fraud_neighbor_counts = []
    for txn_idx in txn_indices:
        # Find accounts linked to this transaction
        accts = txn_to_accts.get(txn_idx, [])
        # Find all other transactions from these accounts
        neighbors = set()
        for acct in accts:
            for neighbor_txn in acct_to_txns.get(acct, []):
                if neighbor_txn != txn_idx:
                    neighbors.add(neighbor_txn)
        # Count how many neighbors are fraud
        fraud_count = sum(1 for n in neighbors if n in fraud_set)
        fraud_neighbor_counts.append(fraud_count)
    
    return fraud_neighbor_counts

print("Counting fraud neighbors (this may take a minute)...")
hybrid_only_neighbors = count_fraud_neighbors(only_hybrid_global[:200])  # sample for speed
both_catch_neighbors = count_fraud_neighbors(both_catch_global[:200])

print(f"\nFraud caught ONLY by Hybrid — avg fraud neighbors: {np.mean(hybrid_only_neighbors):.2f}")
print(f"Fraud caught by BOTH models — avg fraud neighbors: {np.mean(both_catch_neighbors):.2f}")
print(f"\n>> Hybrid-only fraud cases have {'MORE' if np.mean(hybrid_only_neighbors) > np.mean(both_catch_neighbors) else 'FEWER'} fraud neighbors on average")
print(">> This suggests the graph structure helps detect fraud through relational patterns!")

In [ ]:
# Visualize the difference in fraud neighbor counts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Histogram comparison ---
max_val = max(max(hybrid_only_neighbors, default=0), max(both_catch_neighbors, default=0))
bins = range(0, int(max_val) + 2)

axes[0].hist(both_catch_neighbors, bins=bins, alpha=0.6, color='#2ECC71', label='Both Catch', density=True)
axes[0].hist(hybrid_only_neighbors, bins=bins, alpha=0.6, color='#E74C3C', label='Only Hybrid', density=True)
axes[0].set_xlabel('Number of Fraud Neighbors (same account)')
axes[0].set_ylabel('Density')
axes[0].set_title('Fraud Neighbor Distribution')
axes[0].legend()
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# --- Right: Score comparison for the "only hybrid" group ---
# Show how XGBoost scored these vs how Hybrid scored them
hybrid_only_xgb_scores = xgb_scores[only_hybrid_positions]
hybrid_only_hybrid_scores = hybrid_test_scores[only_hybrid_positions]

axes[1].scatter(hybrid_only_xgb_scores, hybrid_only_hybrid_scores,
                alpha=0.4, s=20, color='#E74C3C', label='Only Hybrid catches')
axes[1].axhline(y=hybrid_thresh, color='#E74C3C', linestyle='--', alpha=0.5, label=f'Hybrid threshold ({hybrid_thresh:.2f})')
axes[1].axvline(x=xgb_thresh, color='#5DADE2', linestyle='--', alpha=0.5, label=f'XGBoost threshold ({xgb_thresh:.2f})')
axes[1].set_xlabel('XGBoost Score')
axes[1].set_ylabel('Hybrid GAT+VAE Score')
axes[1].set_title('Fraud Cases: Only Hybrid Catches\n(XGBoost scores too low, Hybrid scores high enough)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('../results/figures/complementarity_details.png', dpi=150)
plt.show()

## 9. Save Model

In [ ]:
os.makedirs('../results/models', exist_ok=True)
torch.save(model.state_dict(), '../results/models/hybrid_gatvae_v2.pt')
print('Model saved: results/models/hybrid_gatvae_v2.pt')

print(f'\nTraining times:')
print(f'  XGBoost:       {xgb_time:.1f}s')
print(f'  GCN:           {gcn_time:.1f}s')
print(f'  Hybrid P1:     {p1_time:.1f}s')
print(f'  Hybrid P2:     {p2_time:.1f}s')
print(f'  Hybrid Total:  {p1_time + p2_time:.1f}s')